In [21]:
import geopandas as gpd
import pandas as pd
import re
import unicodedata


In [19]:
print(data['CATEGORIA'].unique())

print(data['ESTADO'].unique())


<StringArray>
[  'I-1',   'I-2',   'I-3',   'I-4',  'II-E',  'II-2',  'II-1',     '0',
 'III-1', 'III-E', 'III-2']
Length: 11, dtype: str
<StringArray>
[                    'ACTIVO',            'BAJA DEFINITIVA',
  'CIERRE TEMPORAL DE OFICIO',           'BAJA PROVISIONAL',
   'CIERRE TEMPORAL DE PARTE', 'BAJA PROVISIONAL DE OFICIO',
  'BAJA DEFINITIVA DE OFICIO']
Length: 7, dtype: str


In [23]:
import re
import unicodedata
import pandas as pd

# =========================================================================
# 1. Carga
# =========================================================================

RUTA = r'C:\Users\angelo\Documents\GitHub\Tarea_2\data\RENIPRESS_30-04-2026.csv'

data = pd.read_csv(RUTA, sep=";", encoding="latin-1")


# =========================================================================
# 2. Reglas de clasificación (declaradas explícitamente)
# =========================================================================

DEPTOS = {"JUNIN", "CUSCO", "LORETO"}

# Un establecimiento es RESOLUTIVO si y solo si: estado activo Y categoria II/III
CATEGORIAS_RESOLUTIVAS = {"II-1", "II-2", "II-E", "III-1", "III-2", "III-E"}

# Primer nivel: se mantienen en el dataset, se excluyen del calculo
CATEGORIAS_NO_RESOLUTIVAS = {"I-1", "I-2", "I-3", "I-4"}

# '0' es el marcador de ausencia de categoria en RENIPRESS
SIN_CATEGORIA = {"0", "SIN CATEGORIA"}

ESTADOS_ACTIVOS = {"ACTIVO"}

# Bajas y cierres: se mantienen en el dataset, se excluyen del calculo.
# Decision: los cierres temporales NO cuentan como activos.
ESTADOS_NO_ACTIVOS = {
    "BAJA DEFINITIVA",
    "BAJA DEFINITIVA DE OFICIO",
    "BAJA PROVISIONAL",
    "BAJA PROVISIONAL DE OFICIO",
    "CIERRE TEMPORAL DE OFICIO",
    "CIERRE TEMPORAL DE PARTE",
    "SIN ESTADO",
}


# =========================================================================
# 3. Normalizacion
# =========================================================================

def limpiar_texto(valor):
    """Mayusculas, sin tildes, espacios internos colapsados, sin bordes."""
    s = unicodedata.normalize("NFKD", str(valor))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).strip().upper()


def normalizar_estado(valor):
    if pd.isna(valor):
        return "SIN ESTADO"
    return limpiar_texto(valor)


def normalizar_categoria(valor):
    """Categoria canonica ('II-1'), 'SIN CATEGORIA', o None si no se reconoce."""
    if pd.isna(valor):
        return "SIN CATEGORIA"
    s = limpiar_texto(valor)
    if s in {"", "-", "SIN CATEGORIA", "SIN CATEGORIZAR"}:
        return "SIN CATEGORIA"
    if s == "0":
        return "0"
    # unificar guiones unicode, quitar puntos y espacios internos
    s = re.sub(r"[\u2010-\u2015\u2212]", "-", s).replace(".", "").replace(" ", "")
    # reconstruir el guion si falta: "II1" -> "II-1"
    m = re.match(r"^(I{1,3})-?([1-4]|E)$", s)
    return f"{m.group(1)}-{m.group(2)}" if m else None


# =========================================================================
# 4. Aplicacion
# =========================================================================

# Unico filtro que elimina filas: el alcance geografico del estudio
data_filt = data[data["DEPARTAMENTO"].map(limpiar_texto).isin(DEPTOS)].copy()

data_filt["ESTADO_NORM"] = data_filt["ESTADO"].map(normalizar_estado)
data_filt["CATEGORIA_NORM"] = data_filt["CATEGORIA"].map(normalizar_categoria)

data_filt["ES_ACTIVO"] = data_filt["ESTADO_NORM"].isin(ESTADOS_ACTIVOS)
data_filt["CAT_RESOLUTIVA"] = data_filt["CATEGORIA_NORM"].isin(CATEGORIAS_RESOLUTIVAS)

data_filt["ES_RESOLUTIVO"] = data_filt["ES_ACTIVO"] & data_filt["CAT_RESOLUTIVA"]


# =========================================================================
# 5. Verificacion
# =========================================================================

cats_conocidas = CATEGORIAS_RESOLUTIVAS | CATEGORIAS_NO_RESOLUTIVAS | SIN_CATEGORIA
est_conocidos = ESTADOS_ACTIVOS | ESTADOS_NO_ACTIVOS

cats_nuevas = set(data_filt["CATEGORIA_NORM"].dropna()) - cats_conocidas
est_nuevos = set(data_filt["ESTADO_NORM"]) - est_conocidos
no_mapeadas = data_filt.loc[data_filt["CATEGORIA_NORM"].isna(), "CATEGORIA"].unique()

assert not cats_nuevas, f"Categorias no declaradas: {cats_nuevas}"
assert not est_nuevos, f"Estados no declarados: {est_nuevos}"
assert len(no_mapeadas) == 0, f"Categorias sin mapear: {no_mapeadas}"

print(f"Filas en el ambito de estudio: {len(data_filt)}")
print(f"Resolutivos (activos + II/III): {data_filt['ES_RESOLUTIVO'].sum()}")
print()
print(data_filt.groupby("DEPARTAMENTO")["ES_RESOLUTIVO"].agg(["size", "sum"]))


# =========================================================================
# 6. Vista para el calculo de establecimiento mas cercano
# =========================================================================

# data_filt conserva TODAS las filas del ambito; el calculo usa solo esta vista
resolutivos = data_filt[data_filt["ES_RESOLUTIVO"]].copy()

Filas en el ambito de estudio: 4047
Resolutivos (activos + II/III): 68

              size  sum
DEPARTAMENTO           
CUSCO         1577   27
JUNIN         1434   27
LORETO        1036   14


In [26]:
import numpy as np

# =========================================================================
# 7. Diagnostico de coordenadas (puntos 1, 2 y 3)
# =========================================================================

# Bounding box de Peru segun enunciado
LAT_MIN, LAT_MAX = -18.4, -0.04
LON_MIN, LON_MAX = -81.4, -68.6

# Tolerancia para "cero": el archivo trae -1.4e-07, que NO es == 0
EPS = 1e-3

lat = data_filt["NORTE"]
lon = data_filt["ESTE"]

nula = lat.isna() | lon.isna()
cero = (lat.abs() < EPS) | (lon.abs() < EPS)

en_bbox = lat.between(LAT_MIN, LAT_MAX) & lon.between(LON_MIN, LON_MAX)
# Los rangos de lat y lon en Peru no se solapan, asi que un swap es detectable:
# el par invertido cae dentro del bbox y el original no.
en_bbox_swap = lon.between(LAT_MIN, LAT_MAX) & lat.between(LON_MIN, LON_MAX)

# Clasificacion unica y excluyente, en orden de prioridad
data_filt["PROBLEMA_COORD"] = np.select(
    [nula, cero, en_bbox, en_bbox_swap],
    ["NULA", "CERO", "OK", "INVERTIDA"],
    default="FUERA_BBOX",
)

# --- Reparacion de swaps -------------------------------------------------
# Un intercambio lat/lon es inequivoco y recuperable: se corrige y se deja
# registrado. Lo demas no se repara, se descarta.
swap = data_filt["PROBLEMA_COORD"] == "INVERTIDA"

data_filt["LAT"] = np.where(swap, lon, lat)
data_filt["LON"] = np.where(swap, lat, lon)

data_filt["COORD_VALIDA"] = data_filt["PROBLEMA_COORD"].isin(["OK", "INVERTIDA"])


# --- Reporte -------------------------------------------------------------

print("=== Calidad de coordenadas (todo el ambito) ===")
print(data_filt["PROBLEMA_COORD"].value_counts())
print()

print("=== Impacto sobre los resolutivos ===")
res = data_filt[data_filt["ES_RESOLUTIVO"]]
print(res["PROBLEMA_COORD"].value_counts())
print()
print(res.groupby("DEPARTAMENTO")["COORD_VALIDA"].agg(
    total="size", validos="sum"
))
print()

descartados = res[~res["COORD_VALIDA"]]
if len(descartados):
    print("=== Resolutivos descartados por coordenada ===")
    print(descartados[["DEPARTAMENTO", "CATEGORIA_NORM",
                       "NORTE", "ESTE", "PROBLEMA_COORD"]].to_string())

=== Calidad de coordenadas (todo el ambito) ===
PROBLEMA_COORD
OK      2859
NULA    1187
CERO       1
Name: count, dtype: int64

=== Impacto sobre los resolutivos ===
PROBLEMA_COORD
OK      66
NULA     2
Name: count, dtype: int64

              total  validos
DEPARTAMENTO                
CUSCO            27       26
JUNIN            27       26
LORETO           14       14

=== Resolutivos descartados por coordenada ===
      DEPARTAMENTO CATEGORIA_NORM  NORTE  ESTE PROBLEMA_COORD
12724        CUSCO           II-E    NaN   NaN           NULA
22024        JUNIN           II-E    NaN   NaN           NULA


In [27]:
# =========================================================================
# 8. Codigos duplicados (punto 5)
# =========================================================================

# RENIPRESS usa distintos nombres segun la version del archivo
CANDIDATOS_COD = ["Código Único", "Codigo Unico", "CODIGO_UNICO",
                  "COD_UNICO", "Cod_Unico", "codigo_unico", "COD_IPRESS"]

col_cod = next((c for c in CANDIDATOS_COD if c in data_filt.columns), None)
if col_cod is None:
    print("Columnas disponibles:", list(data_filt.columns))
    raise KeyError("Ajusta CANDIDATOS_COD con el nombre real del codigo")

dups = data_filt[data_filt.duplicated(col_cod, keep=False)].sort_values(col_cod)

print(f"=== Codigos duplicados en '{col_cod}' ===")
print(f"Filas involucradas: {len(dups)}  |  codigos distintos: {dups[col_cod].nunique()}")
if len(dups):
    print(dups[[col_cod, "DEPARTAMENTO", "CATEGORIA_NORM",
                "ESTADO_NORM", "NORTE", "ESTE"]].to_string())

# Regla de desempate: conservar el registro activo; si empatan, el primero.
# Se aplica SOLO a la vista de calculo, el dataset completo no se toca.
data_filt["ES_DUPLICADO_DESCARTADO"] = (
    data_filt.sort_values("ES_ACTIVO", ascending=False)
             .duplicated(col_cod, keep="first")
             .reindex(data_filt.index)
)


# =========================================================================
# 9. Encoding (punto 6)
# =========================================================================

# Verificacion directa: si el archivo decodifica como UTF-8 estricto, ES UTF-8
with open(RUTA, "rb") as f:
    crudo = f.read()

try:
    crudo.decode("utf-8")
    encoding_real = "utf-8"
except UnicodeDecodeError:
    encoding_real = "latin-1"

print(f"=== Encoding detectado: {encoding_real} ===")

# Rastro de mojibake: aparece si se leyo UTF-8 como latin-1
MOJIBAKE = r"[ÃÂ�]|â€|Ã±|Ã³|Ã­"

cols_texto = data_filt.select_dtypes(include="object").columns
for c in cols_texto:
    n = data_filt[c].astype(str).str.contains(MOJIBAKE, regex=True, na=False).sum()
    if n:
        print(f"  {c}: {n} filas con caracteres sospechosos")

=== Codigos duplicados en 'COD_IPRESS' ===
Filas involucradas: 0  |  codigos distintos: 0
=== Encoding detectado: utf-8 ===
  NOMBRE: 289 filas con caracteres sospechosos
  CLASIFICACION: 14 filas con caracteres sospechosos
  TIPO_ESTABLECIMIENTO: 431 filas con caracteres sospechosos
  PROVINCIA: 72 filas con caracteres sospechosos
  DISTRITO: 10 filas con caracteres sospechosos
  DIRECCION: 820 filas con caracteres sospechosos
  RED: 71 filas con caracteres sospechosos
  MICRORED: 8 filas con caracteres sospechosos
  UNIDAD_EJECUTORA: 71 filas con caracteres sospechosos
  TELEFONO: 1 filas con caracteres sospechosos
  HORARIO: 12 filas con caracteres sospechosos


C:\Users\angelo\AppData\Local\Temp\ipykernel_27464\2700626647.py:50: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_texto = data_filt.select_dtypes(include="object").columns
